# 第13章：大规模预训练工程

## 本章目标
- 理解为什么「80%的预训练是数据工程」
- 掌握数据清洗、混合比例、FP8 训练、Multi-token 预测等关键技术
- 了解 DeepSeek-V3 等模型背后的工程实践

## 前置知识
- 复习第3章：预训练基础
- 复习第4章：分布式训练

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch matplotlib numpy
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

## 1. 数据清洗流水线

大规模预训练数据的典型处理流程：

```
原始网页数据 → 去重 (MinHash) → 语言过滤 → 质量分类 → PII 去除 → 困惑度过滤 → 预训练数据
```

### MinHash 去重原理

目标：高效判断两个文档是否「近似重复」，不需要逐一比较所有文档对。

步骤：
1. **Shingling**: 把文档拆成 k-gram 子串集合
2. **MinHash 签名**: 对每个 shingle 用 N 个哈希函数，取每个函数的最小值
3. **Jaccard 估计**: 两个文档 MinHash 签名中相同值的比例 ≈ Jaccard 相似度

参考：[FineWeb](https://huggingface.co/datasets/HuggingFaceFW/fineweb), [RefinedWeb](https://arxiv.org/abs/2306.01116) (Penedo et al., 2023)

### MinHash 算法详解

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

Jaccard 相似度直接计算需要 $O(|A| \cdot |B|)$，对百万级文档不可行。

MinHash 的核心性质：$P(\min_{h \in H}(h(A)) = \min_{h \in H}(h(B))) = J(A, B)$

即两个集合的最小哈希值相同的概率等于它们的 Jaccard 相似度。用 N 个独立哈希函数，统计相同比例即可估计 Jaccard。

In [ ]:
import hashlib
import numpy as np
from collections import defaultdict

class MinHashDedup:
    """MinHash 近似去重。
    
    用 N 个哈希函数生成文档的紧凑签名，
    通过签名相似度估计 Jaccard 相似度。
    """
    def __init__(self, num_hashes=128, shingle_size=5):
        self.num_hashes = num_hashes
        self.shingle_size = shingle_size
        # 生成 num_hashes 个不同的哈希种子
        self.seeds = [hashlib.md5(str(i).encode()).hexdigest() for i in range(num_hashes)]

    def _shingling(self, text):
        """将文本拆成 k-gram 集合"""
        return set(text[i:i+self.shingle_size] for i in range(len(text) - self.shingle_size + 1))

    def _hash_shingle(self, shingle, seed):
        """用指定种子哈希一个 shingle"""
        return int(hashlib.sha256((seed + shingle).encode()).hexdigest(), 16)

    def signature(self, text):
        """计算文档的 MinHash 签名"""
        shingles = self._shingling(text)
        if not shingles:
            return [float('inf')] * self.num_hashes
        sig = []
        for seed in self.seeds:
            min_hash = min(self._hash_shingle(s, seed) for s in shingles)
            sig.append(min_hash)
        return sig

    def jaccard_estimate(self, sig1, sig2):
        """通过签名估计 Jaccard 相似度"""
        return sum(a == b for a, b in zip(sig1, sig2)) / self.num_hashes

    def exact_jaccard(self, text1, text2):
        """精确 Jaccard（用于验证）"""
        s1 = self._shingling(text1)
        s2 = self._shingling(text2)
        return len(s1 & s2) / len(s1 | s2) if s1 | s2 else 0

In [ ]:
# 演示：5 个文档的近似去重
docs = [
    "The quick brown fox jumps over the lazy dog in the park on a sunny day",
    "The quick brown fox jumps over the lazy dog in the park on a sunny day",          # 完全重复
    "The quick brown fox jumps over the lazy dog in the garden on a rainy day",        # 高度相似
    "A completely different document about machine learning and neural networks",       # 完全不同
    "Machine learning and neural networks are transforming technology today",           # 主题相似但文本不同
]

dedup = MinHashDedup(num_hashes=128, shingle_size=5)
sigs = [dedup.signature(doc) for doc in docs]

print("文档间相似度矩阵（MinHash 估计 vs 精确 Jaccard）:\n")
for i in range(len(docs)):
    for j in range(i+1, len(docs)):
        est = dedup.jaccard_estimate(sigs[i], sigs[j])
        exact = dedup.exact_jaccard(docs[i], docs[j])
        flag = " ← 近似重复!" if est > 0.5 else ""
        print(f"  doc{i} vs doc{j}: 估计={est:.3f}  精确={exact:.3f}{flag}")

print(f"\n结论：设定阈值 > 0.5，doc0 和 doc1/doc2 被标记为近似重复")

## 2. 数据混合比例

### 为什么混合比例很重要

预训练数据通常包含多个领域：网页文本、代码、数学、书籍、论文等。不同领域的比例直接影响模型能力：

- **代码比例高** → 更好的推理和编程能力
- **数学比例高** → 更好的逻辑推理
- **网页比例高** → 更好的通用知识和对话能力

### DoReMi 算法

用一个小 proxy 模型自动学习最优领域权重：
1. 训练一个小模型在各领域上的困惑度
2. 根据困惑度差异调整领域权重（困惑度高的领域给更多权重）
3. 用最优权重训练大模型

参考：[DoReMi](https://arxiv.org/abs/2305.10429)

### DeepSeek-V3 的数据组成

DeepSeek-V3 使用了约 14.8T tokens 的训练数据，其中代码和数学占比显著高于一般模型：
- 这直接解释了为什么 DeepSeek 在编程和数学推理上表现突出
- 参考：[DeepSeek-V3](https://arxiv.org/abs/2412.19437) Section 3.1

## 3. FP8 训练

### FP8 两种格式

| 格式 | Sign | Exponent | Mantissa | 动态范围 | 精度 |
|------|------|----------|----------|----------|------|
| E4M3 | 1 bit | 4 bits | 3 bits | 较小 (~±448) | 较高 |
| E5M2 | 1 bit | 5 bits | 2 bits | 较大 (~±57344) | 较低 |

**使用策略：**
- Forward pass 用 **E4M3**（需要更高精度保存激活值）
- Backward pass 用 **E5M2**（需要更大动态范围处理梯度）

### 缩放因子粒度
- **Per-tensor**: 整个张量一个缩放因子，最简单但精度损失最大
- **Per-block (tile)**: 按 1×128 或 128×128 分块，精度和效率的平衡
- **Per-channel**: 按输出通道，常用于权重

参考：[FP8 Formats for Deep Learning](https://arxiv.org/abs/2209.05433) (Micikevicius et al., 2022), [DeepSeek-V3](https://arxiv.org/abs/2412.19437) Section 3.3

In [ ]:
import torch
import torch.nn.functional as F

def simulate_fp8_quantize(tensor, format='E4M3'):
    """模拟 FP8 量化（简化版，仅展示核心概念）。
    
    实际 FP8 硬件有更复杂的舍入和异常值处理。
    """
    if format == 'E4M3':
        max_val = 448.0   # E4M3 最大值
        # 3 bit mantissa → 2^3 = 8 个量化级别（每个 2 的幂区间内）
        mantissa_bits = 3
    else:  # E5M2
        max_val = 57344.0  # E5M2 最大值
        mantissa_bits = 2

    # 简化：对称量化到 [-max_val, max_val]
    scale = max_val / tensor.abs().max().clamp(min=1e-8)
    scaled = tensor * scale
    # 模拟有限精度：四舍五入到整数后除以 scale
    quantized = torch.round(scaled) / scale
    # Clamp 到范围
    quantized = quantized.clamp(-max_val, max_val)
    return quantized, scale

# 演示：矩阵乘法的 FP8 量化误差
A = torch.randn(64, 64)  # 模拟激活
B = torch.randn(64, 64)  # 模拟权重

# FP32 基准
result_fp32 = A @ B

# FP8 量化
A_e4m3, scale_a = simulate_fp8_quantize(A, 'E4M3')
B_e4m3, scale_b = simulate_fp8_quantize(B, 'E4M3')
result_fp8 = A_e4m3 @ B_e4m3

# 误差分析
error = (result_fp32 - result_fp8).abs()
relative_error = error / result_fp32.abs().clamp(min=1e-8)

print(f"FP32 结果范围: [{result_fp32.min():.4f}, {result_fp32.max():.4f}]")
print(f"FP8  结果范围: [{result_fp8.min():.4f}, {result_fp8.max():.4f}]")
print(f"绝对误差: mean={error.mean():.6f}, max={error.max():.4f}")
print(f"相对误差: mean={relative_error.mean():.4%}, max={relative_error.max():.4%}")
print(f"\nFP8 的价值：训练速度提升 ~2x（H100 上），代价是可接受的精度损失")

## 4. Multi-token Prediction

### 核心思想

标准语言模型：每个位置只预测**下一个** token → 1 个训练信号/位置

Multi-token prediction：每个位置同时预测**未来 N 个** token → N 个训练信号/位置

**优势：**
1. 更丰富的训练信号（不浪费中间表示）
2. 迫使模型做更长程的规划
3. 实验证明提升下游任务表现，尤其是代码和推理

参考：[Better & Faster Large Language Models via Multi-token Prediction](https://arxiv.org/abs/2404.19737) (Gloeckle et al., 2024)

In [ ]:
import torch
import torch.nn as nn

class MultiTokenGPT(nn.Module):
    """Multi-token prediction：共享 backbone + N 个独立预测头。
    
    这里简化为 2-token prediction（预测 next + next-next token）。
    """
    def __init__(self, vocab_size, n_embd=64, n_head=4, n_layer=2, block_size=64, n_predict=2):
        super().__init__()
        self.n_predict = n_predict
        self.block_size = block_size
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)
        
        # 共享 Transformer backbone
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=n_embd, nhead=n_head, batch_first=True)
            for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        
        # N 个独立的预测头
        self.heads = nn.ModuleList([
            nn.Linear(n_embd, vocab_size) for _ in range(n_predict)
        ])

    def forward(self, idx):
        B, T = idx.size()
        pos = torch.arange(T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)
        
        # Causal mask
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=idx.device)
        for layer in self.layers:
            x = layer(x, src_mask=mask)
        x = self.ln_f(x)
        
        # 每个头预测不同距离的 token
        logits_list = [head(x) for head in self.heads]
        return logits_list

# 演示
vocab_size = 100
model = MultiTokenGPT(vocab_size)
x = torch.randint(0, vocab_size, (4, 32))
logits_list = model(x)

for i, logits in enumerate(logits_list):
    print(f"Head {i} (预测第 {i+1} 个未来 token): logits shape = {logits.shape}")

# 训练时：损失 = 各头的加权平均
def multi_token_loss(logits_list, targets_list, weights=None):
    """计算 multi-token prediction 的总损失。
    
    targets_list[i] 对应 head[i] 预测的第 (i+1) 个未来 token。
    """
    if weights is None:
        weights = [1.0 / len(logits_list)] * len(logits_list)
    total_loss = 0
    for logits, targets, w in zip(logits_list, targets_list, weights):
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        total_loss += w * loss
    return total_loss

print(f"\nMultiTokenGPT 参数量: {sum(p.numel() for p in model.parameters()):,}")

## 5. 课程学习

### 核心思想

模仿人类学习过程：从简单到困难，从已知到未知。

**常见策略：**
1. **难度递增**：先训简单文本（如维基百科），再训复杂文本（如数学证明）
2. **领域渐进**：先训通用语料，再逐步加入代码、数学等专业领域
3. **上下文长度渐进**：先训短序列（4K），再逐渐增长（32K → 128K）

DeepSeek-V3 在训练过程中动态调整数据领域比例，结合多阶段训练策略。

参考：[DeepSeek-V3](https://arxiv.org/abs/2412.19437) 训练细节

## 练习

1. 修改 MinHash 的 `num_hashes` 参数（32, 64, 128, 256），观察估计精度变化
2. 在 `MultiTokenGPT` 中添加第 3 个预测头（预测第 3 个未来 token），观察损失变化
3. 思考：如果你的预训练数据中代码占 60%，可能会带来什么副作用？

## 延伸阅读

- [FineWeb 数据集](https://huggingface.co/datasets/HuggingFaceFW/fineweb) — 大规模清洗后的预训练数据
- [RefinedWeb](https://arxiv.org/abs/2306.01116) — RefinedWeb 数据处理方法 (Penedo et al., 2023)
- [DoReMi](https://arxiv.org/abs/2305.10429) — 最优数据混合比例学习
- [FP8 Formats](https://arxiv.org/abs/2209.05433) — FP8 训练格式 (Micikevicius et al., 2022)
- [Multi-token Prediction](https://arxiv.org/abs/2404.19737) — 多 token 预测 (Gloeckle et al., 2024)
- [DeepSeek-V3](https://arxiv.org/abs/2412.19437) — 大规模预训练工程实践